# California Residential Home Price Prediction
# Baseline Model - Linear Regression

This notebook trains a Linear Regression model as the performance baseline for comparison 
against more complex models (Random Forest, XGBoost) built in later notebooks.

**Evaluation metrics:** R², MAPE, MdAPE

## Section 1 - Setup & Load Data

In [31]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import median_absolute_error, mean_absolute_percentage_error
from sklearn.metrics import r2_score

In [32]:
df_model = pd.read_csv('data/cleaned_data.csv')
print('Shape:', df_model.shape)

Shape: (397461, 23)


## Section 2 - Train Test Split

In [33]:
latest_month = df_model['CloseYear'] * 100 + df_model['CloseMonth']
test_month = latest_month.max()
print(f'Test month: {test_month}')

test_mask = latest_month == test_month
df_train = df_model[~test_mask].copy()
df_test = df_model[test_mask].copy()

X_train = df_train.drop(columns=['ClosePrice'])
y_train = df_train['ClosePrice']
X_test = df_test.drop(columns=['ClosePrice'])
y_test = df_test['ClosePrice']

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

Test month: 202606
Training set: (384874, 22)
Test set: (12587, 22)


### Handling Missing Values for Linear Regression

Note: Linear Regression cannot handle missing values natively, unlike XGBoost. BedBathRatio has 143 missing values in the training set, corresponding to properties with zero recorded bathrooms. These are imputed with the median calculated from the training set only, and the same value is applied to the test set to avoid data leakage.

In [34]:
median_ratio = X_train['BedBathRatio'].median()
X_train['BedBathRatio'] = X_train['BedBathRatio'].fillna(median_ratio)
X_test['BedBathRatio'] = X_test['BedBathRatio'].fillna(median_ratio)

## Seciton 3 - Model Training

Linear Regression is trained on the full feature set with no regularization, scaling, or tuning. This is intentionally the simplest possible baseline.

In [35]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Model trained.')

Model trained.


## Section 4 - Model Evaluation

The model is evaluated using R squared, MAPE, and MdAPE on the held out test month.

In [36]:
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
mdape = np.median(np.abs((y_test - y_pred) / y_test))

print(f'R2: {r2:.4f}')
print(f'MAPE: {mape:.4f}')
print(f'MdAPE: {mdape:.4f}')

R2: 0.5468
MAPE: 0.4000
MdAPE: 0.2700


## Section 5 - Summary

Linear Regression baseline results on the held out test month (June 2026):

- R2: 0.5468
- MAPE: 0.4000
- MdAPE: 0.2700

These results reflect the limitation of a linear model on data where raw numeric features show weak correlation with ClosePrice, as identified in EDA. Location driven pricing patterns and non-linear feature interactions are expected to be captured more effectively by Random Forest and XGBoost in later notebooks.